Cell 1 — Install dependencies

In [ ]:
# ── Install dependencies ───────────────────────────────────────
!pip install transformers datasets torch scikit-learn pandas requests -q

Cell 2 — Konfigurasi

In [ ]:
# ── Konfigurasi — ISI BAGIAN INI ──────────────────────────────
import os

# URL backend Flask kamu (ngrok jika lokal, atau domain jika sudah deploy)
BACKEND_URL = "https://your-backend.com/api"  # ganti ini

# API key — harus sama persis dengan COLAB_API_KEY di .env backend
COLAB_API_KEY = "your-secret-colab-api-key"  # ganti ini

# Direktori kerja di Colab
WORK_DIR = "/content/transmotion"
os.makedirs(WORK_DIR, exist_ok=True)

# Header default untuk semua request ke backend
HEADERS = {
    "X-Colab-Key": COLAB_API_KEY,
    "Content-Type": "application/json",
}

print("✅ Konfigurasi selesai")
print(f"   Backend : {BACKEND_URL}")
print(f"   Work dir: {WORK_DIR}")

Cell 3 — Helper functions

In [ ]:
# ── Helper functions ───────────────────────────────────────────
import requests
import json
import time
import io
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

def api_get(path, params=None):
    """GET request ke backend."""
    try:
        res = requests.get(
            f"{BACKEND_URL}{path}",
            headers={k: v for k, v in HEADERS.items() if k != "Content-Type"},
            params=params,
            timeout=30,
        )
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ GET {path} gagal: {e}")
        return None


def api_post(path, data=None, json_data=None, files=None):
    """POST request ke backend."""
    try:
        headers = {k: v for k, v in HEADERS.items() if k != "Content-Type"}
        if files:
            # multipart — jangan set Content-Type (requests handle otomatis)
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                data=data or {},
                files=files,
                timeout=60,
            )
        else:
            headers["Content-Type"] = "application/json"
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                json=json_data or data or {},
                timeout=30,
            )
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ POST {path} gagal: {e}")
        try:
            print(f"   Response: {e.response.text[:300] if e.response else 'no response'}")
        except Exception:
            pass
        return None


def set_seed(seed=42):
    """Reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def check_gpu():
    """Cek ketersediaan GPU."""
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"✅ GPU tersedia: {gpu_name} ({gpu_mem:.1f} GB)")
        return True
    else:
        print("⚠️  GPU tidak tersedia, menggunakan CPU (training akan lambat)")
        return False


print("✅ Helper functions siap")
check_gpu()

Cell 4 — Cek koneksi ke backend

In [ ]:
# ── Test koneksi ke backend ────────────────────────────────────
def test_connection():
    try:
        res = requests.get(f"{BACKEND_URL.replace('/api', '')}/health", timeout=10)
        if res.status_code == 200:
            print("✅ Backend terhubung:", res.json().get("message"))
            return True
        else:
            print(f"❌ Backend merespons dengan status {res.status_code}")
            return False
    except Exception as e:
        print(f"❌ Tidak bisa terhubung ke backend: {e}")
        print("   Pastikan:")
        print("   1. Backend Flask sudah berjalan")
        print("   2. BACKEND_URL sudah benar")
        print("   3. Jika lokal, pastikan ngrok sudah aktif")
        return False

test_connection()

Cell 5 — Dataset loader

In [ ]:
# ── Dataset loader ─────────────────────────────────────────────
def load_dataset_from_backend(job):
    """
    Ambil preprocessed dataset dari backend via API.
    Mengembalikan DataFrame dengan kolom: preprocessed_text, label
    """
    dataset_id = job["dataset_id"]
    print(f"📦 Mengambil dataset: {job.get('dataset_name', dataset_id)}")

    all_rows = []
    page = 1
    per_page = 500

    while True:
        res = api_get(
            f"/datasets/{dataset_id}/preprocessed",
            params={"page": page, "per_page": per_page},
        )
        if not res or not res.get("data"):
            break

        all_rows.extend(res["data"])

        pagination = res.get("meta", {}).get("pagination", {})
        total_pages = pagination.get("total_pages", 1)

        print(f"   Halaman {page}/{total_pages} — {len(all_rows)} baris diambil...", end="\r")

        if page >= total_pages:
            break
        page += 1

    print(f"\n✅ Total {len(all_rows)} baris diambil dari backend")

    if not all_rows:
        raise ValueError("Dataset kosong atau gagal diambil")

    df = pd.DataFrame(all_rows)
    df = df[["preprocessed_text", "label"]].dropna()
    df = df[df["preprocessed_text"].str.strip() != ""]
    df = df.reset_index(drop=True)

    print(f"   Distribusi kelas:")
    for label, count in df["label"].value_counts().items():
        pct = count / len(df) * 100
        print(f"   - {label}: {count} ({pct:.1f}%)")

    return df


def create_stratified_split(df, test_size, seed=42):
    """
    Buat train/test split stratified berdasarkan split_info dari job.
    """
    from sklearn.model_selection import train_test_split

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=seed,
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"\n📊 Split hasil:")
    print(f"   Train : {len(train_df)} baris")
    print(f"   Test  : {len(test_df)} baris")

    return train_df, test_df


print("✅ Dataset loader siap")

Cell 6 — HuggingFace Dataset & Model setup

In [ ]:
# ── HuggingFace Dataset & Tokenizer setup ─────────────────────
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)


MODEL_BASE_NAMES = {
    "mbert": "bert-base-multilingual-cased",
    "xlmr": "xlm-roberta-base",
}


def build_label_maps(labels):
    """Buat mapping label ↔ integer."""
    unique_labels = sorted(set(labels))
    label2id = {l: i for i, l in enumerate(unique_labels)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label


def tokenize_dataset(df, tokenizer, label2id, max_length):
    """Konversi DataFrame ke HuggingFace Dataset dengan tokenisasi."""
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    hf_data = HFDataset.from_dict({
        "text": df["preprocessed_text"].tolist(),
        "labels": [label2id[l] for l in df["label"].tolist()],
    })

    tokenized = hf_data.map(
        tokenize_fn,
        batched=True,
        remove_columns=["text"],
        desc="Tokenisasi",
    )
    return tokenized


def compute_metrics(eval_pred):
    """Metrik evaluasi untuk Trainer."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
    }


print("✅ HuggingFace setup siap")
print(f"   PyTorch version : {torch.__version__}")
print(f"   CUDA available  : {torch.cuda.is_available()}")

Cell 7 — Progress reporter ke backend

In [ ]:
# ── Progress reporter ──────────────────────────────────────────
from transformers import TrainerCallback

class BackendProgressCallback(TrainerCallback):
    """
    Callback yang melaporkan progress training ke backend
    setiap akhir epoch.
    """

    def __init__(self, job_id, total_epochs, colab_session_id=None):
        self.job_id = job_id
        self.total_epochs = total_epochs
        self.colab_session_id = colab_session_id
        self.current_epoch = 0

    def on_epoch_end(self, args, state, control, **kwargs):
        self.current_epoch = int(state.epoch)
        progress = int((self.current_epoch / self.total_epochs) * 100)

        # Ambil metrik dari log terakhir
        log = state.log_history[-1] if state.log_history else {}
        train_loss = log.get("loss") or log.get("train_loss")
        val_loss = log.get("eval_loss")
        val_accuracy = log.get("eval_accuracy")
        val_f1 = log.get("eval_f1")

        payload = {
            "current_epoch": self.current_epoch,
            "total_epochs": self.total_epochs,
            "progress": progress,
            "train_loss": round(float(train_loss), 4) if train_loss else None,
            "val_loss": round(float(val_loss), 4) if val_loss else None,
            "val_accuracy": round(float(val_accuracy), 4) if val_accuracy else None,
            "val_f1": round(float(val_f1), 4) if val_f1 else None,
            "colab_session_id": self.colab_session_id,
        }

        result = api_post(f"/colab/jobs/{self.job_id}/progress", json_data=payload)
        if result:
            print(
                f"\n📡 Epoch {self.current_epoch}/{self.total_epochs} dilaporkan — "
                f"loss: {train_loss:.4f if train_loss else 'N/A'}, "
                f"val_acc: {val_accuracy:.4f if val_accuracy else 'N/A'}"
            )
        else:
            print(f"\n⚠️  Gagal melaporkan progress epoch {self.current_epoch}")


print("✅ Progress callback siap")

Cell 8 — Training function utama

In [ ]:
# ── Training function utama ────────────────────────────────────
def train_model(job):
    """
    Jalankan fine-tuning berdasarkan job dari backend.
    """
    set_seed(42)

    job_id = job["id"]
    model_type = job["model_type"]
    hp = job["hyperparams"]
    split_info = job.get("split_info", {})

    epochs = hp.get("epochs", 3)
    batch_size = hp.get("batch_size", 16)
    max_length = hp.get("max_length", 128)
    learning_rate = hp.get("learning_rate", 2e-5)
    warmup_steps = hp.get("warmup_steps", 0)
    weight_decay = hp.get("weight_decay", 0.01)
    test_size = split_info.get("test_size", 0.2)

    base_model = MODEL_BASE_NAMES.get(model_type, "bert-base-multilingual-cased")

    print("=" * 55)
    print(f"🚀 Memulai Training Job: {job_id[:8]}...")
    print(f"   Model      : {base_model}")
    print(f"   Epochs     : {epochs}")
    print(f"   Batch size : {batch_size}")
    print(f"   Max length : {max_length}")
    print(f"   LR         : {learning_rate}")
    print(f"   Test size  : {test_size * 100:.0f}%")
    print("=" * 55)

    # ── 1. Load dataset ────────────────────────────────────────
    df = load_dataset_from_backend(job)
    train_df, test_df = create_stratified_split(df, test_size)

    label2id, id2label = build_label_maps(df["label"])
    num_labels = len(label2id)
    print(f"\n🏷️  Labels ({num_labels}): {list(label2id.keys())}")

    # ── 2. Load tokenizer & model ──────────────────────────────
    print(f"\n📥 Memuat tokenizer & model: {base_model}...")
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"   Device: {device.upper()}")

    # ── 3. Tokenisasi ──────────────────────────────────────────
    print("\n🔤 Tokenisasi dataset...")
    train_dataset = tokenize_dataset(train_df, tokenizer, label2id, max_length)
    eval_dataset = tokenize_dataset(test_df, tokenizer, label2id, max_length)

    print(f"   Train tokens: {len(train_dataset)}")
    print(f"   Eval tokens : {len(eval_dataset)}")

    # ── 4. Training arguments ──────────────────────────────────
    output_dir = os.path.join(WORK_DIR, f"job_{job_id[:8]}")
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=10,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2 if torch.cuda.is_available() else 0,
    )

    # ── 5. Trainer ─────────────────────────────────────────────
    colab_session_id = f"colab_{int(time.time())}"

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            BackendProgressCallback(
                job_id=job_id,
                total_epochs=epochs,
                colab_session_id=colab_session_id,
            ),
            EarlyStoppingCallback(early_stopping_patience=2),
        ],
    )

    # ── 6. Train ───────────────────────────────────────────────
    print("\n🏋️  Training dimulai...")
    train_result = trainer.train()

    # ── 7. Evaluasi final ──────────────────────────────────────
    print("\n📊 Evaluasi akhir pada test set...")
    eval_results = trainer.evaluate()

    # Hitung metrik detail
    predictions_output = trainer.predict(eval_dataset)
    y_pred = predictions_output.predictions.argmax(axis=-1)
    y_true = test_df["label"].map(label2id).values

    accuracy = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    precision = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    recall = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))

    print(f"\n   Accuracy  : {accuracy * 100:.2f}%")
    print(f"   F1 Score  : {f1 * 100:.2f}%")
    print(f"   Precision : {precision * 100:.2f}%")
    print(f"   Recall    : {recall * 100:.2f}%")

    print("\n📋 Classification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=list(id2label.values()),
        zero_division=0,
    ))

    # ── 8. Simpan model weights ────────────────────────────────
    model_path = os.path.join(output_dir, "model_weights.pt")
    print(f"\n💾 Menyimpan model ke {model_path}...")

    # Simpan hanya state_dict (lebih ringan dari full model)
    torch.save(model.state_dict(), model_path)
    model_size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"   Ukuran file: {model_size_mb:.1f} MB")

    # ── 9. Upload ke backend ───────────────────────────────────
    print("\n📤 Mengupload model ke backend...")

    job_name = job.get("job_name") or f"{model_type.upper()} Model"
    model_name = f"{job_name} — {time.strftime('%Y%m%d_%H%M')}"

    label_map_json = json.dumps({str(i): l for i, l in id2label.items()})

    with open(model_path, "rb") as model_file:
        result = api_post(
            f"/colab/jobs/{job_id}/complete",
            data={
                "model_name": model_name,
                "accuracy": str(accuracy),
                "f1_score": str(f1),
                "precision": str(precision),
                "recall": str(recall),
                "label_map": label_map_json,
                "base_model_name": base_model,
                "colab_session_id": colab_session_id,
            },
            files={"model_file": (f"model_{job_id[:8]}.pt", model_file, "application/octet-stream")},
        )

    if result and result.get("success"):
        print("✅ Model berhasil diupload ke backend!")
        print(f"   Model ID: {result['data']['id']}")
        print(f"   Nama    : {result['data']['name']}")
    else:
        print("❌ Gagal mengupload model ke backend")
        print(f"   Response: {result}")

    # Cleanup output dir (hemat storage Colab)
    import shutil
    shutil.rmtree(output_dir, ignore_errors=True)

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "model_name": model_name,
    }


print("✅ Training function siap")

Cell 9 — Error handler

In [ ]:
# ── Error handler ──────────────────────────────────────────────
def report_failure(job_id, error_message):
    """Laporkan kegagalan ke backend."""
    print(f"\n❌ Melaporkan kegagalan ke backend: {error_message[:200]}")
    result = api_post(
        f"/colab/jobs/{job_id}/fail",
        json_data={"error_message": str(error_message)[:1000]},
    )
    if result:
        print("   Status kegagalan berhasil dilaporkan")
    else:
        print("   ⚠️  Gagal melaporkan kegagalan ke backend")


print("✅ Error handler siap")

Cell 10 — Main loop (jalankan cell ini untuk mulai)

In [ ]:
# ── MAIN LOOP ──────────────────────────────────────────────────
# Jalankan cell ini untuk memulai Colab sebagai worker training.
# Biarkan cell ini berjalan selama kamu ingin Colab standby menerima job.
# ──────────────────────────────────────────────────────────────

POLL_INTERVAL_SECONDS = 10  # cek job baru setiap N detik
MAX_CONSECUTIVE_ERRORS = 5  # berhenti jika N kali error berturut-turut

print("🤖 Colab Worker aktif")
print(f"   Polling setiap {POLL_INTERVAL_SECONDS} detik...")
print(f"   Backend: {BACKEND_URL}")
print("   Tekan ■ Stop untuk menghentikan\n")

consecutive_errors = 0
total_jobs_done = 0

while True:
    try:
        # ── Poll job berikutnya ────────────────────────────────
        response = api_get("/colab/jobs/next")

        if response is None:
            consecutive_errors += 1
            print(f"⚠️  Tidak bisa menghubungi backend ({consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print("❌ Terlalu banyak error berturut-turut. Worker berhenti.")
                print("   Periksa koneksi dan restart cell ini.")
                break
            time.sleep(POLL_INTERVAL_SECONDS * 2)
            continue

        consecutive_errors = 0  # reset setelah berhasil konek

        job = response.get("data")

        if not job:
            # Tidak ada job — tunggu
            print(
                f"💤 Tidak ada job [{time.strftime('%H:%M:%S')}] — "
                f"menunggu {POLL_INTERVAL_SECONDS}s...",
                end="\r",
            )
            time.sleep(POLL_INTERVAL_SECONDS)
            continue

        # ── Ada job — mulai training ───────────────────────────
        print(f"\n🔔 Job baru ditemukan: {job['id'][:8]}... ({job.get('display_name', '')})")
        total_jobs_done_before = total_jobs_done

        try:
            metrics = train_model(job)
            total_jobs_done += 1
            print(f"\n🎉 Job selesai! Total job diproses: {total_jobs_done}")
            print(
                f"   Accuracy: {metrics['accuracy'] * 100:.2f}% | "
                f"F1: {metrics['f1'] * 100:.2f}%\n"
            )

        except KeyboardInterrupt:
            print("\n⏹️  Training dihentikan manual")
            report_failure(job["id"], "Training dihentikan oleh user")
            raise  # propagate keluar dari while loop

        except Exception as train_err:
            import traceback
            error_msg = traceback.format_exc()
            print(f"\n❌ Training gagal: {train_err}")
            report_failure(job["id"], error_msg)
            print("   Melanjutkan polling untuk job berikutnya...\n")
            time.sleep(5)

    except KeyboardInterrupt:
        print("\n\n⏹️  Worker dihentikan")
        print(f"   Total job berhasil diproses: {total_jobs_done}")
        break

    except Exception as e:
        print(f"\n⚠️  Error di main loop: {e}")
        consecutive_errors += 1
        time.sleep(POLL_INTERVAL_SECONDS)

Cell 11 — Utilitas tambahan (opsional)

In [ ]:
# ── Utilitas Tambahan ──────────────────────────────────────────
# Jalankan cell-cell ini secara manual jika diperlukan.

# --- Cek status job tertentu ---
def check_job_status(job_id):
    """Cek status job dari backend."""
    # endpoint ini menggunakan JWT bukan API key, gunakan untuk debugging saja
    try:
        res = requests.get(
            f"{BACKEND_URL}/training-jobs/{job_id}",
            headers={"X-Colab-Key": COLAB_API_KEY},
            timeout=10,
        )
        data = res.json()
        job = data.get("data", {})
        print(f"Job ID    : {job.get('id')}")
        print(f"Status    : {job.get('status')}")
        print(f"Progress  : {job.get('progress')}%")
        print(f"Epoch     : {job.get('current_epoch')}/{job.get('total_epochs')}")
    except Exception as e:
        print(f"Error: {e}")

# --- Lihat daftar job yang menunggu ---
def list_queued_jobs():
    """Lihat berapa job yang sedang menunggu."""
    res = api_get("/colab/jobs/next")
    if res and res.get("data"):
        job = res["data"]
        print(f"✅ Ada job menunggu: {job['id'][:8]} — {job.get('display_name')}")
    else:
        print("💤 Tidak ada job yang menunggu saat ini")

# --- Bersihkan cache model di Colab ---
def cleanup_work_dir():
    """Hapus semua file kerja di WORK_DIR."""
    import shutil
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
        os.makedirs(WORK_DIR)
        print(f"✅ Work dir dibersihkan: {WORK_DIR}")
    else:
        print("Work dir tidak ada")

print("✅ Utilitas siap")
print("   - check_job_status(job_id)")
print("   - list_queued_jobs()")
print("   - cleanup_work_dir()")

Cell 12 — Mode single job (alternatif loop)

In [ ]:
# ── Mode Single Job ────────────────────────────────────────────
# Gunakan ini jika kamu hanya ingin menjalankan SATU job saja
# tanpa loop terus-menerus.

def run_single_job():
    """Ambil dan jalankan satu job berikutnya, lalu berhenti."""
    print("🔍 Mencari job berikutnya...")

    response = api_get("/colab/jobs/next")

    if not response:
        print("❌ Tidak bisa menghubungi backend")
        return

    job = response.get("data")
    if not job:
        print("💤 Tidak ada job yang menunggu")
        return

    print(f"✅ Job ditemukan: {job['id'][:8]} — {job.get('display_name')}")

    try:
        metrics = train_model(job)
        print(f"\n🎉 Selesai!")
        print(f"   Accuracy : {metrics['accuracy'] * 100:.2f}%")
        print(f"   F1 Score : {metrics['f1'] * 100:.2f}%")
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"❌ Gagal: {e}")
        report_failure(job["id"], error_msg)

# Uncomment untuk jalankan:
# run_single_job()

PETUNJUK PENGGUNAAN

Cara menghubungkan Colab ke backend lokal (ngrok)

Jika backend Flask kamu jalan di komputer lokal, tambahkan cell ini di bagian paling atas notebook sebelum Cell 2:

Setup ngrok (hanya jika backend lokal)
Jalankan ini DI MESIN LOKAL kamu (bukan di Colab):
1. pip install pyngrok
2. ngrok config add-authtoken YOUR_NGROK_TOKEN
3. ngrok http 5000
Salin URL yang muncul (cth: https://abc123.ngrok-free.app)
ke variabel BACKEND_URL di Cell 2 di bawah.
Atau jalankan ngrok dari terminal:
  ngrok http 5000
Tidak perlu install apapun di Colab untuk ini.

print("💡 Pastikan ngrok sudah aktif di mesin lokal dan URL sudah disalin ke BACKEND_URL")
```

### Cara menggunakan notebook
```
1. Buka Google Colab
2. Runtime → Change runtime type → GPU (T4)
3. Jalankan Cell 1 (install deps) — tunggu selesai
4. Isi BACKEND_URL dan COLAB_API_KEY di Cell 2
5. Jalankan Cell 2 sampai Cell 9 satu per satu
6. Jalankan Cell 10 (main loop) — biarkan berjalan

Sekarang Colab standby menunggu job.
Buat job dari website → Colab otomatis ambil dan proses.

Update be/.env.example
Tambahkan COLAB_API_KEY yang lebih jelas:
Colab Integration
Generate dengan: python -c "import secrets; print(secrets.token_hex(32))"

Wajib sama persis dengan yang di-set di Cell 2 Colab notebook
COLAB_API_KEY=ganti_dengan_string_acak_yang_panjang_dan_aman